# Курс «Продвинутое обучение с подкреплением» @ AI Masters
## Домашнее задание 6: Decision Transformer с памятью

### Введение

Decision Transformer (DT) — мощный алгоритм для оффлайн-обучения с подкреплением, обладающий значительной гибкостью для модификаций. Один из распространённых подходов — добавление механизмов памяти для улучшения его возможностей.

Хотя DT изначально способен справляться с частично наблюдаемыми задачами Маркова (POMDP) благодаря механизму внимания, внедрение явных структур памяти может помочь обрабатывать зависимости, выходящие за пределы типового контекста внимания.

В этом задании предлагается реализовать и оценить Decision Transformer с расширенной памятью для задач управления в POMDP.
> **Примечание:** Решения для GRU и LSTM будут добавлены после дедлайна.

### Установка

Перед началом установите необходимые зависимости:

```bash
pip install torch numpy matplotlib gymnasium tqdm
```

### 1. POMDP-окружения

Используются классические задачи MDP из Gymnasium (CartPole, Pendulum, MountainCar), превращённые в POMDP с помощью специальных обёрток, которые вводят частичную наблюдаемость:

- `velocity_cartpole.py`: CartPole с скрытой информацией о скорости
- `flickering_pendulum.py`: Pendulum со случайно пропадающими наблюдениями
- `lidar_mountain_car.py`: MountainCar только с лидарами (датчиками расстояния)

**Важное замечание:** Код тщательно протестирован только с `VelocityCartPoleEnv`. Эксперименты с другими окружениями требуют дополнительных изменений:
- `FlickeringPendulumEnv`: Необходимо добавить поддержку непрерывных пространств действий
- `LiDARMountainCarEnv`: Необходимо реализовать методы сбора траекторий

В этом задании основной акцент на `VelocityCartPoleEnv`. Хотя Decision Transformer с длиной контекста > 1 должен решить эту задачу, цель — исследовать, как механизмы памяти могут её улучшить.

### 2. Сбор оффлайн-датасета RL

Поскольку мы работаем в оффлайн-режиме RL, сначала нужно собрать обучающие данные:

```bash
python train_and_collect_data.py --env velocity_cartpole --train_timesteps 300000 --num_trajectories 100 --reward_threshold 475
```

Это обучит агента PPO-GRU, который затем будет использоваться для генерации траекторий.

После сбора датасета проверьте производительность агента PPO-GRU:

```bash
python utils/visualize_ppo_agent.py --env velocity_cartpole --model_path pomdp_datasets/velocity_cartpole/recurrent_ppo_velocity_cartpole.pt --rnn_type gru
```

И ознакомьтесь со статистикой датасета:

```bash
python memory_dt.py --dataset pomdp_datasets/velocity_cartpole --stats_only
```

Вы должны увидеть примерно такой вывод:

```
Статистика датасета:
Всего эпизодов: 100
Всего шагов: 49924
Средняя награда за эпизод: 499.24
Медианная награда за эпизод: 500.00
Мин./Макс. награда: 476.00/500.00
Стандартное отклонение награды: 3.62
Средняя длина эпизода: 499.24
Процентили награды:
  10%: 500.00
  25%: 500.00
  50%: 500.00
  75%: 500.00
  90%: 500.00
  95%: 500.00
  99%: 500.00
```

Датасет содержит высококачественные траектории со средними наградами, близкими к максимуму для CartPole (500).

### 3. Decision Transformer с памятью

Сначала обучите и проверьте стандартный Decision Transformer в качестве базовой линии:

```bash
# Обучение базового DT
python run_memory_dt.py --env velocity_cartpole --memory_type none --n_epochs 7 --eval_episodes 20

# Проверка базового DT
python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_None_best.pt --memory_type none 
```

Сохраните результаты для дальнейшего сравнения с моделями, усиленными памятью.

### Задания

#### Задание 1: Реализация GRU и LSTM памяти (0.2 балла)

Рекуррентные нейронные сети, такие как LSTM и GRU, — естественный выбор для реализации памяти в нейронных сетях. Хотя Decision Transformer работает только с вниманием, добавление рекуррентной памяти — интересное направление.

Дополните код в `memory_dt.py` для реализации памяти на GRU и LSTM:

```python
# файл: memory_dt.py
self.pos_encoder = PositionalEncoding(n_embed)

# Память
if memory_type == 'gru':
   # TODO: Реализовать память на GRU
   self.memory_proj = nn.Linear(memory_dim, n_embed)
elif memory_type == 'lstm':
   # TODO: Реализовать память на LSTM
   self.memory_proj = nn.Linear(memory_dim, n_embed)
else:
   self.memory = None

# Transformer
```

```python
# файл: memory_dt.py
# добавить память
if self.memory is not None:
   if self.memory_type == 'gru':
         if self.hidden_state is None:
            # TODO: Реализовать память на GRU
         
         memory_out, self.hidden_state = self.memory(state_embeddings, self.hidden_state)
   elif self.memory_type == 'lstm':
         if self.hidden_state is None:
            # TODO: Реализовать память на LSTM
         
         memory_out, self.hidden_state = self.memory(state_embeddings, self.hidden_state)
   
   # проекция памяти в размерность эмбеддинга
   memory_embedding = self.memory_proj(memory_out)
```

После реализации модулей памяти обучите улучшенные модели:

```bash
# Обучение DT+GRU (0.1 балла)
python run_memory_dt.py --env velocity_cartpole --memory_type gru --n_epochs 7 --eval_episodes 20

# Обучение DT+LSTM (0.1 балла)
python run_memory_dt.py --env velocity_cartpole --memory_type lstm --n_epochs 7 --eval_episodes 20
```

Проверьте их производительность:

```bash
# Проверка DT+GRU
python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_gru_best.pt --memory_type gru 

# Проверка DT+LSTM
python utils/visualize_dt_agent.py --env velocity_cartpole --model_path models/memory_dt_velocity_cartpole_lstm_best.pt --memory_type lstm 
```

#### Задание 2: Сравнительный анализ (0.3 балла)

Сравните производительность стандартного DT с реализациями DT+GRU и DT+LSTM. Проанализируйте:

- Эффективность обучения (скорость сходимости)
- Итоговые метрики производительности
- Объём данных, необходимый для сходимости
- Общий эффект добавления модулей памяти

Проведите эксперименты с различными параметрами моделей (длина контекста, размер обучающей выборки, качество данных и др.) и сравните результаты. Что происходит при изменении return-to-go (RTG)?

Приложите графики обучения и метрики для подтверждения анализа.

#### Задание 3: Собственный механизм памяти (0.3 балла)

Разработайте и реализуйте собственный механизм памяти для Decision Transformer. Здесь нет единственно правильного ответа — проявите креативность в разумных пределах.

Опишите ваш подход:
- Какую информацию будет обрабатывать ваш механизм?
- Как информация будет обрабатываться?
- Какую архитектуру модели вы используете?
- Какой подход к обучению выбран?

Проведите эксперименты и сравните вашу реализацию с предыдущими подходами. Какие выводы можно сделать?

### Правила сдачи (0.2 балла)

1. Форкните или склонируйте этот репозиторий и внесите изменения
2. Добавьте отчёт с результатами экспериментов и анализом в любом удобном формате (.ipynb, .pdf и т.д.) в репозиторий
3. Отправьте файл со ссылкой на ваш репозиторий в телеграм-бот для сдачи

Хорошо структурированный, профессионально оформленный отчёт приносит 0.2 балла.

### Структура репозитория

- `pomdp_envs/velocity_cartpole.py`: Реализация CartPole с сокрытием скорости
- `pomdp_envs/flickering_pendulum.py`: Реализация Pendulum с мерцающими наблюдениями
- `pomdp_envs/lidar_mountain_car.py`: Реализация MountainCar с лидарами
- `recurrent_ppo.py`: Реализация рекуррентного PPO для сбора данных
- `train_and_collect_data.py`: Скрипт обучения агента PPO и сбора траекторий
- `memory_dt.py`: Реализация Decision Transformer с памятью
- `run_memory_dt.py`: Скрипт для обучения и оценки Decision Transformer с памятью